# CLO DUAL JSON MAKER COVER ARC DATA 
Centerpoint 기준으로 arrangement point하지 않고

각 패널의 Mean Point를 기준으로 arrangement point 생성

++ 추가적으로 ARC 2 Bezier data에 대해서도 다룰 수 있게 됨.ㅋ.ㅋ

++ 내부의 Internal Line 데이터까지 추가 하는 Version 2025.02.18 기준

In [6]:
import os, sys
sys.path.append(os.path.dirname(os.getcwd()))
from glob import glob
import math
from pprint import pprint
import torch

import pygarment as pyg
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time

from tqdm import tqdm
import json
import random

import trimesh
from ANALYSIS.analysis_utils import (
    plot_panel_info,
    visualize_meshes_plotly,
    v_id_map
)


In [7]:
# TODO: 2025.03.03 front, back body 부분 배치포인트 1단계씩 올림.
batch_points = {
"Arm_Back_1_L" : [ 20.4240,  132.30371, -15.25501],
"Arm_Back_2_L" : [ 25.4240,  127.80371, -15.25501],
"Arm_Back_3_L" : [ 30.4240,  123.30371, -15.25501],
"Arm_Back_4_L" : [ 35.4240,  118.80371, -15.25501],
"Arm_Back_5_L" : [ 40.4240,  114.30371, -15.25501],
"Arm_Back_6_L" : [ 45.4240,  109.80371, -15.25501],
"Arm_Back_7_L" : [ 50.4240,  105.30371, -15.25501],
"Arm_Back_8_L" : [ 55.4240,  100.80371, -15.25501],
"Arm_Back_1_R" : [ -20.4240,  132.30371, -15.25501],
"Arm_Back_2_R" : [ -25.4240,  127.80371, -15.25501],
"Arm_Back_3_R" : [ -30.4240,  123.30371, -15.25501],
"Arm_Back_4_R" : [ -35.4240,  118.80371, -15.25501],
"Arm_Back_5_R" : [ -40.4240,  114.30371, -15.25501],
"Arm_Back_6_R" : [ -45.4240,  109.80371, -15.25501],
"Arm_Back_7_R" : [ -50.4240,  105.30371, -15.25501],
"Arm_Back_8_R" : [ -55.4240,  100.80371, -15.25501],
"Arm_Front_1_L" : [ 20.4240,  132.30371, 15.25501],
"Arm_Front_2_L" : [ 25.4240,  127.80371, 15.25501],
"Arm_Front_3_L" : [ 30.4240,  123.30371, 15.25501],
"Arm_Front_4_L" : [ 35.4240,  118.80371, 15.25501],
"Arm_Front_5_L" : [ 40.4240,  114.30371, 15.25501],
"Arm_Front_6_L" : [ 45.4240,  109.80371, 15.25501],
"Arm_Front_7_L" : [ 50.4240,  105.30371, 15.25501],
"Arm_Front_8_L" : [ 55.4240,  100.80371, 15.25501],
"Arm_Front_1_R" : [ -20.4240,  132.30371, 15.25501],
"Arm_Front_2_R" : [ -25.4240,  127.80371, 15.25501],
"Arm_Front_3_R" : [ -30.4240,  123.30371, 15.25501],
"Arm_Front_4_R" : [ -35.4240,  118.80371, 15.25501],
"Arm_Front_5_R" : [ -40.4240,  114.30371, 15.25501],
"Arm_Front_6_R" : [ -45.4240,  109.80371, 15.25501],
"Arm_Front_7_R" : [ -50.4240,  105.30371, 15.25501],
"Arm_Front_8_R" : [ -55.4240,  100.80371, 15.25501],
"Body_Front_Center_1" :[  0., 142.30371, 15.25501],
"Body_Front_Center_2" : [  0., 135.30371, 15.25501],
"Body_Front_Center_3" : [  0., 128.30371, 15.25501],
"Body_Front_Center_4" : [  0., 121.30371, 15.25501],
"Body_Front_Center_5" : [  0., 114.30371, 15.25501],
"Body_Front_Center_6" : [  0., 107.30371, 15.25501],
"Body_Front_Center_7" : [  0., 100.30371, 15.25501],
"Body_Front_Center_8" : [  0., 93.30371, 15.25501],
"Body_Front_Center_9" : [  0., 79.30371, 15.25501],
"Body_Back_Center_1" : [  0., 142.30371,-15.25501],
"Body_Back_Center_2" : [  0., 135.30371,-15.25501],
"Body_Back_Center_3" : [  0., 128.30371,-15.25501],
"Body_Back_Center_4" : [  0., 121.30371,-15.25501],
"Body_Back_Center_5" : [  0., 114.30371,-15.25501],
"Body_Back_Center_6" : [  0., 107.30371,-15.25501],
"Body_Back_Center_7" : [  0., 100.30371,-15.25501],
"Body_Back_Center_8" : [  0., 93.30371, -15.25501],
"Body_Back_Center_9" : [  0., 86.30371, -15.25501],
"Body_Back_Center_10": [  0., 75.30371, -15.25501],
#"Body_Front_1_R" : [ -11.86359, 142.30371,   15.25501],
"Body_Front_1_R" : [ -11.86359, 135.30371,   15.25501],
"Body_Front_2_R" : [ -11.86359, 128.30371,   15.25501],
"Body_Front_3_R" : [ -11.86359, 121.30371,   15.25501],
"Body_Front_4_R" : [ -11.86359, 114.30371,   15.25501],
"Body_Front_5_R" : [ -11.86359, 107.30371,   15.25501],
"Body_Front_6_R" : [ -11.86359, 100.30371,   15.25501],
#"Body_Front_1_L" : [ 11.86359, 142.30371,   15.25501],
"Body_Front_1_L" : [ 11.86359, 135.30371,   15.25501],
"Body_Front_2_L" : [ 11.86359, 128.30371,   15.25501],
"Body_Front_3_L" : [ 11.86359, 121.30371,   15.25501],
"Body_Front_4_L" : [ 11.86359, 114.30371,   15.25501],
"Body_Front_5_L" : [ 11.86359, 107.30371,   15.25501],
"Body_Front_6_L" : [ 11.86359, 100.30371,   15.25501],
#"Body_Back_1_R" :[ -11.86359, 142.30371,   -15.25501],
"Body_Back_1_R" :[ -11.86359, 135.30371,   -15.25501],
"Body_Back_2_R" :[ -11.86359, 128.30371,   -15.25501],
"Body_Back_3_R" :[ -11.86359, 121.30371,   -15.25501],
"Body_Back_4_R" :[ -11.86359, 114.30371,   -15.25501],
"Body_Back_5_R" :[ -11.86359, 107.30371,   -15.25501],
"Body_Back_6_R" :[ -11.86359, 100.30371,   -15.25501],
#"Body_Back_1_L" :[ 11.86359, 142.30371,   -15.25501],
"Body_Back_1_L" :[ 11.86359, 135.30371,   -15.25501],
"Body_Back_2_L" :[ 11.86359, 128.30371,   -15.25501],
"Body_Back_3_L" :[ 11.86359, 121.30371,   -15.25501],
"Body_Back_4_L" :[ 11.86359, 114.30371,   -15.25501],
"Body_Back_5_L" :[ 11.86359, 107.30371,   -15.25501],
"Body_Back_6_L" :[ 11.86359, 100.30371,   -15.25501],
"Leg_Front_1_L" : [ 9.86359, 78.30371,  15.25501], 
"Leg_Front_2_L" : [ 9.86359, 68.30371,  15.25501],
"Leg_Front_3_L" : [ 9.86359, 58.30371,  15.25501],
"Leg_Front_1_R" :[ -9.86359, 78.30371,  15.25501], 
"Leg_Front_2_R" :[ -9.86359, 68.30371,  15.25501],
"Leg_Front_3_R" :[ -9.86359, 58.30371,  15.25501], 
"Leg_Back_1_L" :  [ 9.86359, 78.30371,  -15.25501], 
"Leg_Back_2_L" :  [ 9.86359, 68.30371,  -15.25501],
"Leg_Back_3_L" :  [ 9.86359, 58.30371,  -15.25501],
"Leg_Back_1_R" : [ -9.86359, 78.30371,  -15.25501], 
"Leg_Back_2_R" : [ -9.86359, 68.30371,  -15.25501],
"Leg_Back_3_R" : [ -9.86359, 58.30371,  -15.25501],
"Leg_Front_Mid_1" :[-13.86359, 54.30371, 15.25501],
"Leg_Front_Mid_3" :[13.86359, 54.30371, 15.25501],
"Leg_Back_Mid_1" :[13.86359, 54.30371, -15.25501],
"Leg_Back_Mid_3" :[-13.86359, 54.30371, -15.25501],
"Skirt_Front_Center_1":[0, 50.30371, 15.25501],
"Skirt_Front_Center_2":[0, 42.30371, 15.25501],
"Skirt_Front_Center_3":[0, 36.30371, 15.25501],
"Skirt_Front_Center_4":[0, 30.30371, 15.25501],
"Skirt_Front_Center_5":[0, 18.30371, 15.25501],
"Skirt_Front_Center_6" : [0, 6.30371, 15.25501],
"Skirt_Back_Center" : [0, 54.30371, -15.25501],
"Skirt_Back_Center_1" :[0, 50.30371, -15.25501],
"Skirt_Back_Center_2" :[0, 42.30371, -15.25501],
"Skirt_Back_Center_3" :[0, 36.30371, -15.25501],
"Skirt_Back_Center_4" :[0, 30.30371, -15.25501],
"Skirt_Back_Center_5" :[0, 18.30371, -15.25501],
"Skirt_Back_Center_6" :[0, 6.30371, -15.25501],
"Leg_Front_Mid_L" : [9.86359, 48.30371, 15.25501], 
"Leg_Front_Mid_R" :[-9.86359, 48.30371, 15.25501], 
"Leg_Back_Mid_L" :[9.86359, 48.30371, -15.25501],
"Leg_Back_Mid_R" :[-9.86359, 48.30371, -15.25501],
"Shoulder_Top_L" : [  11.424, 142.30371, 0.],
"Shoulder_Top_R": [  -11.424, 142.30371, 0.],
"Neck_L" :[  5.424, 149.30371, 1.],
"Neck_R" :[  -5.424, 149.30371, 1.],
"Neck_Back_Center" :[  0., 149.30371, -15.25501],
"Neck_Front_Center" :[  0., 145.30371, 15.25501],
#"Head_L" : [ 10., 160.30371, 0],
#"Head_R" : [ -10., 160.30371, 0],
"Head_Back_L" : [ 7., 160.30371, -15.25501],
"Head_Back_R" : [ -7., 160.30371, -15.25501],
"Head_Back_Center" :[ 0., 160.30371, -15.25501],
"Head_Front_Center" : [ 0., 160.30371, 15.25501],
"Head_L" :[ 5., 172.30371, 0], #Head_Top_L  인걸 바꿈
"Head_R" :[ -5., 172.30371, 0], #Head_Top_R  인걸 바꿈
"Head_Top_Center" : [ 0., 172.30371, 0],
"Leg_Front_Point_1" :[18.86359, 22.30371, 15.25501],
"Leg_Front_Point_2" :[11.86359, 22.30371, 15.25501],
"Leg_Front_Point_3" :[4.86359, 22.30371, 15.25501],
"Leg_Front_Point_4" :[-4.86359, 22.30371, 15.25501],
"Leg_Front_Point_5" :[-11.86359, 22.30371, 15.25501],
"Leg_Front_Point_6" :[-18.86359, 22.30371, 15.25501],
"Leg_Back_Point_1" :[18.86359, 22.30371, -15.25501],
"Leg_Back_Point_2" :[11.86359, 22.30371, -15.25501],
"Leg_Back_Point_3" :[4.86359, 22.30371, -15.25501],
"Leg_Back_Point_4" :[-4.86359, 22.30371, -15.25501],
"Leg_Back_Point_5" :[-11.86359, 22.30371, -15.25501], 
"Leg_Back_Point_6" :[-18.86359, 22.30371, -15.25501],
}

for key, value in batch_points.items():
    batch_points[key] = [value[0], value[1] -10, value[2]]

In [7]:
import pandas as pd
from svgpathtools import Path, Line, QuadraticBezier, CubicBezier, Arc
import os
import random
import string
import json

def load_json_file(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

def save_json(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    #print(f"Saved: {filepath}")

def generate_random_hex():
    return ''.join(random.choices('0123456789ABCDEF', k=6))

import math
import numpy as np
import matplotlib.pyplot as plt

def svg_arc_to_center(x1, y1, x2, y2, rx, ry, phi, large_arc, sweep):
    """
    SVG의 endpoint-to-center 변환 알고리즘(회전 phi 포함)을 수행합니다.
    여기서는 phi가 0라고 가정할 수 있으므로, cos(phi)=1, sin(phi)=0를 사용합니다.
    
    반환: (cx, cy, theta_start, theta_end, delta_theta)
    """

    # phi (회전각)을 라디안으로
    phi_rad = math.radians(phi)
    
    # Step 1: 중간점과 변환된 좌표 (phi=0이면 그대로)
    dx = (x1 - x2) / 2.0
    dy = (y1 - y2) / 2.0
    # x1', y1'
    x1p = math.cos(phi_rad)*dx + math.sin(phi_rad)*dy
    y1p = -math.sin(phi_rad)*dx + math.cos(phi_rad)*dy
    
    # Step 2: rx,ry 스케일링 체크 (여기서는 필요 없으리라 가정)
    lam = (x1p**2)/(rx**2) + (y1p**2)/(ry**2)
    if lam > 1:
        scale = math.sqrt(lam)
        rx *= scale
        ry *= scale

    # Step 3: 계수 (SVG 사양에 따른)
    num = rx**2 * ry**2 - rx**2 * y1p**2 - ry**2 * x1p**2
    den = rx**2 * y1p**2 + ry**2 * x1p**2
    # 부동소수점 오차로 음수 방지
    factor = math.sqrt(max(0, num/den)) if den != 0 else 0
    # SVG 사양에 따라, large_arc와 sweep 플래그가 같으면 부호 반전

    if large_arc == sweep:
        factor = -factor
        
        
    cxp = factor * (rx * y1p / ry)
    cyp = factor * (-ry * x1p / rx)

    # Step 4: 원래 좌표계로 복원 (phi=0이면 회전 없이 중간점에 더함)
    mid_x = (x1 + x2) / 2.0
    mid_y = (y1 + y2) / 2.0
    cx = math.cos(phi_rad)*cxp - math.sin(phi_rad)*cyp + mid_x
    cy = math.sin(phi_rad)*cxp + math.cos(phi_rad)*cyp + mid_y

    # Step 5: 시작각과 끝각 (원 중심 기준)
    theta_start = math.atan2(y1 - cy, x1 - cx)
    theta_end   = math.atan2(y2 - cy, x2 - cx)
    
    # Step 6: 아크의 진행각 delta_theta 결정 (SVG 사양)
    delta_theta = theta_end - theta_start
    
    if sweep:
        if delta_theta < 0:
            delta_theta += 2*math.pi
    else:
        if delta_theta > 0:
            delta_theta -= 2*math.pi
    
    # 대개 추가 보정은 필요하나, 여기서는 플래그에 따른 delta_theta만 사용
    return cx, cy, theta_start, theta_end, delta_theta

def reflect_two_points_across_line(P0, P1, P2, P3):
    """
    직선 P0-P3에 대해,
    점 P1과 P2를 '동시에' 대칭시킨 (P1', P2')를 반환한다.
    
    각 인자는 (x, y) 형태의 튜플 또는 리스트라고 가정.
    """
    P0 = np.array(P0, dtype=float)    
    P1 = np.array(P1, dtype=float)
    P2 = np.array(P2, dtype=float)
    P3 = np.array(P3, dtype=float)
    # 직선 방향벡터
    v = P3 - P0
    # 내적 (v·v)
    vdotv = v.dot(v)
    
    # 만약 P0와 P3가 같다면 반사 불가능 → 그냥 원래 점 반환하거나 예외 처리
    if abs(vdotv) < 1e-15:
        return (tuple(P1), tuple(P2))
    
    # ---- P1 대칭 ----
    w1 = P1 - P0
    w1dotv = w1.dot(v)
    w1_proj = (w1dotv / vdotv) * v
    w1_prime = 2 * w1_proj - w1
    P1_reflected = P0 + w1_prime  # P1'
    
    # ---- P2 대칭 ----
    w2 = P2 - P0
    w2dotv = w2.dot(v)
    w2_proj = (w2dotv / vdotv) * v
    w2_prime = 2 * w2_proj - w2
    P2_reflected = P0 + w2_prime  # P2'
    
    return tuple(P1_reflected), tuple(P2_reflected)

def single_arc_to_cubic_bezier(x1, y1, x2, y2, rx, ry, rotation, large_arc, sweep):
    """
    주어진 ARC 정보를 기반으로 단일 Cubic Bézier 곡선으로 근사합니다.
    회전(
    n)은 도 단위이며, 여기서는 rotation=0가 주어졌습니다.
    
    반환: (P0, P1, P2, P3), 그리고 (cx, cy, theta_start, theta_end, delta_theta)
    """
        
    cx, cy, theta_start, theta_end, delta_theta = svg_arc_to_center(x1, y1, x2, y2, rx, ry, rotation, large_arc, sweep)
    
    # 엔드포인트들
    P0 = (cx + rx * math.cos(theta_start), cy + ry * math.sin(theta_start))
    P3 = (cx + rx * math.cos(theta_end),   cy + ry * math.sin(theta_end))
    
    # Cubic Bézier 공식: k = (4/3)*tan(delta_theta/4)
    k = (4.0/3.0) * math.tan(delta_theta/4.0)
    
    # 컨트롤 포인트
    P1 = (P0[0] - k * rx * math.sin(theta_start),
          P0[1] + k * ry * math.cos(theta_start))
    P2 = (P3[0] + k * rx * math.sin(theta_end),
          P3[1] - k * ry * math.cos(theta_end))
    
    return (P0, P1, P2, P3)

def euler_angles_to_rotation_matrix(euler_angles):
    """
    Convert Euler angles to a rotation matrix.
    
    Parameters:
    euler_angles (tuple or list): A tuple or list of three angles (roll, pitch, yaw) in radians.
    
    Returns:
    np.ndarray: A 3x3 rotation matrix.
    """
    roll, pitch, yaw = euler_angles

    # Compute individual rotation matrices
    R_x = np.array([
        [1, 0, 0],
        [0, np.cos(roll), -np.sin(roll)],
        [0, np.sin(roll), np.cos(roll)]
    ])

    R_y = np.array([
        [np.cos(pitch), 0, np.sin(pitch)],
        [0, 1, 0],
        [-np.sin(pitch), 0, np.cos(pitch)]
    ])

    R_z = np.array([
        [np.cos(yaw), -np.sin(yaw), 0],
        [np.sin(yaw), np.cos(yaw), 0],
        [0, 0, 1]
    ])

    # Combine the rotation matrices
    R = np.dot(R_z, np.dot(R_y, R_x))

    return R

def rotation_matrix_2d(euler_am):
    """
    Convert an angle in radians to a 2D rotation matrix.
    
    Parameters:
    theta (float): The rotation angle in radians.
    
    Returns:
    np.ndarray: A 2x2 rotation matrix.
    """
    c = np.cos(euler_am)
    s = np.sin(euler_am)
    
    R = np.array([
        [c, -s],
        [s,  c]
    ])
    
    return R

def rotate_point(x, y, theta):
    """
    
    a 2D point (x, y) by an angle theta.
    
    Parameters:
    x (float): Original x-coordinate.
    y (float): Original y-coordinate.
    theta (float): Rotation angle in radians.
    
    Returns:
    (float, float): Rotated (x', y') coordinates.
    """
    R = rotation_matrix_2d(theta)  # 회전 행렬 계산
    rotated = R @ np.array([x, y]) # 행렬 곱 수행
    
    return float(rotated[0]), float(rotated[1])  # 변환된 좌표 반환


def calculate_center_point(panel_name, spec, panel_svg_path_dict):
    vertices_from_spec = np.asarray(spec['pattern']['panels'][panel_name]['vertices'])    
    offset = np.min(vertices_from_spec * np.array([1, -1]), axis=0)

    euler_angles = spec['pattern']['panels'][panel_name]['rotation']
    translation = spec['pattern']['panels'][panel_name]['translation']
    rotation_matrix = euler_angles_to_rotation_matrix(euler_angles)

    path = panel_svg_path_dict[panel_name][0]
    
    x1, x2, y1, y2 = path.bbox()
    xm = (x1 + x2) / 2
    ym = (y1 + y2) / 2 
    
    bbox_center = np.array([xm, ym])
    bbox_center = (bbox_center + offset) * np.array([1, -1])
    
    batch_point = np.array([
        bbox_center[0], bbox_center[1], 1
    ]) @ rotation_matrix.T + translation
    #print(panel_name, batch_point) 
    return batch_point.reshape(1, -1)
    # batch_point_list.append(batch_point.reshape(1, -1))

def matching_point(target_point):
# 모든 batch_point들과 target_point 간의 거리(L2 norm) 계산
    keys = list(batch_points.keys())  # batch_point의 key 목록
    values = np.array(list(batch_points.values()))  # value를 numpy 배열로 변환

    # L2 거리 계산 (유클리드 거리)
    distances = np.linalg.norm(values - target_point, axis=1)

    # 가장 가까운 key 찾기
    closest_index = np.argmin(distances)  # 최소 거리의 인덱스
    closest_key = keys[closest_index]  # 해당 인덱스의 key 가져오기

    #print(f"가장 가까운 batch_point: {closest_key}")
    
    return closest_key    

def process_path_data(data):
    panel_edge_list = []
    # Path 내부 요소 순회
    for panel_edge in data[0]:
        if panel_edge.__class__.__name__ == "Arc":
            data = Arc_Curve(panel_edge)
            data = data.data_to_dict()
        elif panel_edge.__class__.__name__ == "QuadraticBezier":
            data = QuadraticBezier_Curve(panel_edge)
            data = data.data_to_dict()
        elif panel_edge.__class__.__name__ == "CubicBezier":
            data = CubicBezier_Curve(panel_edge)
            data = data.data_to_dict()
        elif panel_edge.__class__.__name__ == "Line":
            data = Line_Curve(panel_edge)
            data = data.data_to_dict()
        else:
            pass
        
        panel_edge_list.append(data)
    
    panel_edge_list = scale_coordinates(panel_edge_list, scale=8.5)
    
    return panel_edge_list

def calculate_edge_length(edge):
    """
    각 엣지의 길이를 계산. QuadraticBezier의 경우 CubicBezier처럼 처리. 
    """
    
    if edge.__class__.__name__ == "Line":
        return edge.length()
    elif edge.__class__.__name__ == "QuadraticBezier":
        # QuadraticBezier -> CubicBezier 변환
        # 기존 컨트롤 포인트
        control1 = edge.control
        
        # 새로운 컨트롤 포인트 추가
        control2 = complex(round(control1.real, 4), control1.imag)  # x좌표 소수점 4자리로 변경
        # QuadraticBezier -> CubicBezier 변환
        cubic_edge = CubicBezier(
            start=edge.start,
            control1=control1,
            control2=control2,
            end=edge.end
        )
        # CubicBezier 길이 계산
        return float(cubic_edge.length())
    elif edge.__class__.__name__ == "CubicBezier":
        return edge.length()
    elif edge.__class__.__name__ == "Arc":
        return edge.length()
    else:
        raise ValueError(f"Unsupported edge type: {type(edge)}, {edge}")

def calculate_fstart_fend(path):
    """
    전체 엣지에 대해 fstart와 fend를 계산.
    """
    total_length = sum(calculate_edge_length(edge_info) for edge_info in path)
    fstart = 0.0
    results = []

    for idx, edge in enumerate(path):  # enumerate를 사용하여 번호 부여
        edge_length = calculate_edge_length(edge) # TODO: name 넣어봄 01.19.17:00
        fend = fstart + (edge_length / total_length)
        results.append({
            "edge_id": idx,  # 엣지 번호
            "edge": str(edge),  # Edge를 문자열로 변환하여 저장
            "fstart": round(fstart, 6),
            "fend": round(fend, 6)
        })
        fstart = fend

    return results

# 패널 이름에 따라 데이터프레임 처리
def process_negative_columns(panel_name, panel_data):
    # 패널 이름이 '_back', '_b', '_btorso'로 끝나는 경우
    if '_back' in panel_name or '_b' in panel_name or '_btorso' in panel_name or "_b_l" in panel_name or "_b_r" in panel_name:
        for segment in panel_data:
            if "start_x" in segment and isinstance(segment["start_x"], (int, float)):
                segment["start_x"] *= -1
            if "end_x" in segment and isinstance(segment["end_x"], (int, float)):
                segment["end_x"] *= -1

            if "control1_x" in segment and isinstance(segment["control1_x"], (int, float)):
                segment["control1_x"] *= -1
            if "control2_x" in segment and isinstance(segment["control2_x"], (int, float)):
                segment["control2_x"] *= -1
            else:
                pass#print(f"Skipping panel: {panel_name}")
    else:
        pass
    return panel_data

def convert_stitch_dict(stitch_dict_info, panel_index_mapping_info):
    new_stitch_dict = {}

    for key, value in stitch_dict_info.items():
        filtered_list = []
        for stitch in value:
            if isinstance(stitch, dict) and 'panel' in stitch and 'edge' in stitch:
                filtered_list.append({
                    'panel': panel_index_mapping_info[stitch['panel']],
                    'edge': stitch['edge']
                })
            else:
                # dict가 아니거나 panel/edge 키가 없으면 무시
                pass

        new_stitch_dict[key] = filtered_list

    value_stack = []
    for value in new_stitch_dict.values():
        # dictionary stack
        for value in value:
            value_stack.append(value)     
    return value_stack

# 통합할때 추가하는 방법. (2024.02.04 작성)
def unify_clo_json_data(Clo_json_path, Clo_json_path_to_add):
    with open(Clo_json_path, 'r') as file:
        json_data = json.load(file)
    with open(Clo_json_path_to_add, 'r') as file:
        json_data_to_add = json.load(file)
    
    # FabricList, GradingRuleTableList(제껴도됨), PatternList, SymmetricDataList(제껴도됨), InstanceDataList, SeamLinePairGroupList
    # 통합할때 추가하는 방법.
    # Fabric의 경우
    json_data['FabricList'].append(json_data_to_add['FabricList'][0])
    # Pattern의 경우
    [json_data['PatternList'].append(json_data_to_add['PatternList'][num]) for num in range(len(json_data_to_add['PatternList']))]
    # InstanceDataList의 경우
    [json_data['InstanceDataList'].append(json_data_to_add['InstanceDataList'][num]) for num in range(len(json_data_to_add['InstanceDataList']))]
    # SeamLinePairGroupList의 경우
    [json_data['SeamLinePairGroupList'].append(json_data_to_add['SeamLinePairGroupList'][num]) for num in range(len(json_data_to_add['SeamLinePairGroupList']))]
    
    return json_data
# 통합된 STITCH 추가하는 방법. (2024.02.04 작성)
def stitch_data_unify(Clo_stitch_path, Clo_stitch_path_to_add):
    with open(Clo_stitch_path, "r") as f:
        stitch_data_1 = json.load(f)
    #stitch_data_1
    with open(Clo_stitch_path_to_add, "r") as f:
        stitch_data_2 = json.load(f)
    
    max_panel_1 = max(d["panel"] for d in stitch_data_1)
    offset = max_panel_1 + 1
    for item in stitch_data_2:
        item["panel"] += offset
    garment_first_id = os.path.basename(os.path.dirname(Clo_stitch_path))
    garment_second_id = os.path.basename(os.path.dirname(Clo_stitch_path_to_add))

    return (stitch_data_1+stitch_data_2), [garment_first_id, garment_second_id]

# Unique UUID 생성
def get_unique_random_string(existing_set):
    while True:
        # 영문(대소문자) + 숫자
        candidate = ''.join(random.choices(string.ascii_letters + string.digits, k=6))
        # 중복 검사
        if candidate not in existing_set:
            existing_set.add(candidate)
            return candidate
        
# Arc -> Bezier Curve로 변환하는 함수 사용해서 변환.
# 다 float으로 변환해봄.
class Arc_Curve:
    def __init__(self, panel_data):
        self.type = "Arc"
        self.start_x = float(panel_data.start.real)
        self.start_y = float(panel_data.start.imag)
        self.end_x = float(panel_data.end.real)
        self.end_y = float(panel_data.end.imag)
        self.radius = panel_data.radius
        self.rotation = panel_data.rotation
        self.large_arc = panel_data.large_arc
        self.sweep = panel_data.sweep
    def data_to_dict(self):
        return {
            "type": "Arc",
            "start_x": self.start_x,
            "start_y": self.start_y,
            "end_x": self.end_x,
            "end_y": self.end_y,
            "radius": self.radius,
            "rotation": self.rotation,
            "large_arc": self.large_arc,
            "sweep": self.sweep,
        }

class CubicBezier_Curve:
    def __init__(self, panel_data):
        self.type = "Bezier Curve"
        self.start_x = float(panel_data.start.real)
        self.start_y = float(panel_data.start.imag)
        self.control1_x = float(panel_data.control1.real)
        self.control1_y = float(panel_data.control1.imag)
        self.control2_x = float(panel_data.control2.real)
        self.control2_y = float(panel_data.control2.imag)
        self.end_x = float(panel_data.end.real)
        self.end_y = float(panel_data.end.imag)
    def data_to_dict(self):
        return {
            "type": self.type,
            "start_x": self.start_x,
            "start_y": self.start_y,
            "control1_x": self.control1_x,
            "control1_y": self.control1_y,
            "control2_x": self.control2_x,
            "control2_y": self.control2_y,
            "end_x": self.end_x,
            "end_y": self.end_y
        }

class Line_Curve:
    def __init__(self, panel_data):
        self.type = "Straight"
        self.start_x = float(panel_data.start.real)
        self.start_y = float(panel_data.start.imag)
        self.end_x = float(panel_data.end.real)
        self.end_y = float(panel_data.end.imag)
    def data_to_dict(self):
        return {
            "type": self.type,
            "start_x": self.start_x,
            "start_y": self.start_y,
            "end_x": self.end_x,
            "end_y": self.end_y
        }

# QuadraticBezier -> Bezier Curve로        
class QuadraticBezier_Curve:
    def __init__(self, panel_data):
        self.type = "Bezier Curve"
        self.start_x = float(panel_data.start.real)
        self.start_y = float(panel_data.start.imag)
        self.control1_x = float(panel_data.control.real)
        self.control1_y = float(panel_data.control.imag)
        # 여기서는 control2를 추가로 저장하여 QuadraticBezier를 확장함
        self.control2_x = round(panel_data.control.real, 4)
        self.control2_y = round(panel_data.control.imag, 4)
        self.end_x = float(panel_data.end.real)
        self.end_y = float(panel_data.end.imag)
    def data_to_dict(self):
        return {
            "type": self.type,
            "start_x": self.start_x,
            "start_y": self.start_y,
            "control1_x": self.control1_x,
            "control1_y": self.control1_y,
            "control2_x": self.control2_x,
            "control2_y": self.control2_y,
            "end_x": self.end_x,
            "end_y": self.end_y
        }

class ARC_TO_BEZIER:
    def __init__(self, panel_data):
        self.type = "Arc"
        self.start_x = panel_data["start_x"]
        self.start_y = panel_data["start_y"]
        self.end_x = panel_data["end_x"]
        self.end_y = panel_data["end_y"]
        self.radius_x = panel_data["radius"] * 8.5
        self.radius_y = panel_data["radius"] * 8.5
        self.rotation = panel_data["rotation"]
        self.large_arc = panel_data["large_arc"]
        self.sweep = panel_data["sweep"]
        
    def data_to_bezier_curve(self):
        radius_x = float(self.radius_x.real)
        radius_y = float(self.radius_y.imag)
        start, control_1, control_2, end = single_arc_to_cubic_bezier(self.start_x, self.start_y, 
                                self.end_x, self.end_y, 
                                radius_x, radius_y,
                                self.rotation,self.large_arc, self.sweep)
        control_1, control_2 = reflect_two_points_across_line(start, control_1, control_2, end)
        return {
            "type": "Bezier Curve",
            "start_x": self.start_x,
            "start_y": self.start_y,
            "control1_x": float(control_1[0]),
            "control1_y": float(control_1[1]),
            "control2_x": float(control_2[0]),
            "control2_y": float(control_2[1]),
            "end_x": self.end_x,
            "end_y": self.end_y,
        }
        
def scale_coordinates(data_list, scale=8.5):
    # 모든 *_y 값들을 모으기
    y_values = []
    for item in data_list:
        for key, value in item.items():
            if key.endswith("_y"):
                y_values.append(value)
                
    if not y_values:
        raise ValueError("데이터 내에 '_y' 키가 없습니다.")
    
    min_y = min(y_values)
    max_y = max(y_values)
    stand_y = min_y + max_y  # stand_y = (min_y + max_y)
    
    # 각 딕셔너리의 *_x, *_y 값을 scaling 적용
    for item in data_list:
        for key in item.keys():
            if key.endswith("_y"):
                # 새로운 값 = (stand_y - 기존 값) * scale
                item[key] = (stand_y - item[key]) * scale
            elif key.endswith("_x"):
                # 새로운 값 = 기존 값 * scale
                item[key] = item[key] * scale
    return data_list

In [4]:
# Internal Line 그리기 위해서 사용하는 함수.
import math
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, LineString

##############################################################################
# 1. Cubic Bézier 곡선을 폴리라인으로 근사하는 함수
##############################################################################
def approximate_cubic_bezier(p0, c1, c2, p3, steps=10):
    """
    3차 베지어 곡선을 'steps' 개로 분할하여 (x, y) 좌표 리스트를 리턴.
    """
    points = []
    for i in range(steps+1):
        t = i / steps
        x = (1 - t)**3 * p0[0] + 3*(1 - t)**2 * t * c1[0] + 3*(1 - t)*t**2 * c2[0] + t**3 * p3[0]
        y = (1 - t)**3 * p0[1] + 3*(1 - t)**2 * t * c1[1] + 3*(1 - t)*t**2 * c2[1] + t**3 * p3[1]
        points.append((x, y))
    return points

##############################################################################
# 2. 에지 리스트로부터 패널 폴리라인 만들기 (전체 외곽선용)
##############################################################################
def build_polyline_from_edges(edges):
    """
    edges: [
      [(x1, y1, 'Straight'), (x2, y2, 'Straight')],
      [(x3, y3, 'Straight'), (cx1, cy1, 'Bezier Curve'), (cx2, cy2, 'Bezier Curve'), (x4, y4, 'Straight')],
      ...
    ]
    """
    all_points = []
    for segment in edges:
        n = len(segment)
        if n == 2:
            # 직선 세그먼트
            p0 = (segment['points'][0][0], segment['points'][0][1])
            p1 = (segment['points'][1][0], segment['points'][1][1])
            if not all_points:
                all_points.append(p0)
            else:
                if all_points[-1] != p0:
                    all_points.append(p0)
            all_points.append(p1)
        elif n == 4:
            # 3차 베지어 세그먼트
            p0 = (segment['points'][0][0], segment['points'][0][1])
            c1 = (segment['points'][1][0], segment['points'][1][1])
            c2 = (segment['points'][2][0], segment['points'][2][1])
            p3 = (segment['points'][3][0], segment['points'][3][1])
            curve_points = approximate_cubic_bezier(p0, c1, c2, p3, steps=10)
            if not all_points:
                all_points.extend(curve_points)
            else:
                if all_points[-1] == curve_points[0]:
                    all_points.extend(curve_points[1:])
                else:
                    all_points.extend(curve_points)
        else:
            raise ValueError("세그먼트 점 개수가 2(직선) 또는 4(베지어)만 지원합니다.")
    if all_points and (all_points[0] != all_points[-1]):
        all_points.append(all_points[0])
    return all_points

##############################################################################
# 3. 다각형의 외곽선 방향(orientation)을 구하는 헬퍼 함수 (shoelace 공식)
##############################################################################
def compute_polygon_orientation(points):
    """
    points: (x, y) 좌표 리스트 (폐곡선일 경우 마지막 점이 첫 점과 동일)
    리턴: 'ccw' (반시계방향) 또는 'cw' (시계방향)
    """
    area = 0
    # 마지막 중복점은 제외
    for i in range(len(points)-1):
        x0, y0 = points[i]
        x1, y1 = points[i+1]
        area += x0 * y1 - x1 * y0
    return 'ccw' if area > 0 else 'cw'

##############################################################################
# 4. 두 선분(무한 직선)의 교차점을 구하는 헬퍼 함수
##############################################################################
def line_intersection(p1, p2, p3, p4):
    """
    p1, p2: 첫 번째 직선을 정의하는 두 점
    p3, p4: 두 번째 직선을 정의하는 두 점
    리턴: 두 직선의 교차점 (x, y) 또는 평행하면 None
    """
    x1, y1 = p1; x2, y2 = p2
    x3, y3 = p3; x4, y4 = p4
    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if abs(denom) < 1e-10:
        return None  # 평행
    intersect_x = ((x1*y2 - y1*x2) * (x3 - x4) - (x1 - x2) * (x3*y4 - y3*x4)) / denom
    intersect_y = ((x1*y2 - y1*x2) * (y3 - y4) - (y1 - y2) * (x3*y4 - y3*x4)) / denom
    return (intersect_x, intersect_y)

##############################################################################
# 5. 각 패널의 원래 엣지(세그먼트)를 개별적으로 오프셋한 후, 
#    오프셋 결과와 함께 해당 엣지 유형(Bezier/Straight)도 저장하고,
#    인접 엣지의 교차점을 계산해 시작/끝점이 연결되도록 보정하는 함수
##############################################################################

def extract_internal_offset_edges_by_segment(panel_edge_list, offset_dist):    
    """
    panel_edge_list = [{ "start_x": a, "start_y": b, "end_x": c, "end_y": d, "type": "Straight"},
                       { "start_x": a, "start_y": b, "control1_x": e, "control1_y": f, "control2_x": g, "control2_y": h, "end_x": c, "end_y": d, "type": "Bezier Curve"}]
    """
    """
    리턴: 각 패널별로 원래의 각 엣지에 대해,
           {'type': 'Straight' 또는 'Bezier', 'offset_points': [(x, y), ...]} 형태의 리스트
           단, 인접 엣지간의 offset 결과가 교차점을 기준으로 연결됩니다.
    """
    panel_offset_edges = []
    original_edges = []
    # 원래 엣지를 분리 (타입 정보 포함)
    for seg in panel_edge_list:
        if seg["type"] == "Bezier Curve":
            p0 = (seg["start_x"], seg["start_y"])
            c1 = (seg["control1_x"], seg["control1_y"])
            c2 = (seg["control2_x"], seg["control2_y"])
            p3 = (seg["end_x"], seg["end_y"])
            bezier_points = approximate_cubic_bezier(p0, c1, c2, p3)
            original_edges.append({'points': bezier_points, 'type': 'Bezier'})
        elif seg["type"] == "Straight":
            p0 = (seg["start_x"], seg["start_y"])
            p1 = (seg["end_x"], seg["end_y"])
            original_edges.append({'points': [p0, p1], 'type': 'Straight'})
        else:
            pass
            
    # 전체 외곽선으로부터 다각형 방향(orientation) 구하기
    ### 여기부터 변환 필요.
    outline_points = build_polyline_from_edges(original_edges)
    orientation = compute_polygon_orientation(outline_points)
    
    # 각 엣지별 오프셋 계산 (LineString.parallel_offset 사용)
    panel_offset_edges = []
    for edge in original_edges:
        line = LineString(edge['points'])
        # **내부 오프셋: polygon 내부가 CCW이면 interior는 left, CW이면 interior는 right**
        if offset_dist < 0:
            side = 'left' if orientation == 'ccw' else 'right'
        else:
            side = 'right' if orientation == 'ccw' else 'left'
        try:
            offset_line = line.parallel_offset(abs(offset_dist), side=side, resolution=16)
        except Exception as e:
            offset_line = None
        offset_coords = []
        if offset_line is not None:
            if offset_line.is_empty:
                offset_coords = []
            elif offset_line.geom_type == 'LineString':
                offset_coords = list(offset_line.coords)
            elif offset_line.geom_type.startswith('Multi'):
                # MultiLineString: 가장 긴 부분 선택
                parts = list(offset_line)
                longest = max(parts, key=lambda ls: ls.length)
                offset_coords = list(longest.coords)
        panel_offset_edges.append({'type': edge['type'], 'offset_points': offset_coords})
    
    # 보정: 인접 엣지의 offset 결과가 만나도록 (연결된 start/end) 교차점을 계산
    n = len(panel_offset_edges)
    if n > 1:
        # 순환 구조라고 가정 (폐곡선)
        for i in range(n):
            curr_edge = panel_offset_edges[i]
            next_edge = panel_offset_edges[(i+1) % n]
            if not curr_edge['offset_points'] or not next_edge['offset_points']:
                continue
            curr_start = curr_edge['offset_points'][0]
            curr_end = curr_edge['offset_points'][-1]
            next_start = next_edge['offset_points'][0]
            next_end = next_edge['offset_points'][-1]
            inter = line_intersection(curr_start, curr_end, next_start, next_end)
            if inter is not None:
                curr_edge['offset_points'][-1] = inter
                next_edge['offset_points'][0] = inter
    
    # panel_offset_edges.append(panel_offset_edges) 이거 불필요
    return panel_offset_edges



In [5]:
import math
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, LineString
from shapely.ops import unary_union

def approximate_cubic_bezier(p0, c1, c2, p3, steps=10):
    """
    3차 베지어 곡선을 'steps' 개로 분할하여 (x, y) 좌표 리스트를 리턴.
    """
    points = []
    for i in range(steps+1):
        t = i / steps
        x = (1 - t)**3 * p0[0] + 3*(1 - t)**2 * t * c1[0] + 3*(1 - t)*t**2 * c2[0] + t**3 * p3[0]
        y = (1 - t)**3 * p0[1] + 3*(1 - t)**2 * t * c1[1] + 3*(1 - t)*t**2 * c2[1] + t**3 * p3[1]
        points.append((x, y))
    return points

def build_polyline_from_edges(edges):
    """
    edges: [
      {'points': [(x1, y1), (x2, y2)], 'type': 'Straight'},
      {'points': [(x3, y3), (cx1, cy1), (cx2, cy2), (x4, y4)], 'type': 'Bezier'},
      ...
    ]
    -> polyline (폐곡선) 리턴
    """
    all_points = []
    for segment in edges:
        pts = segment['points']
        n = len(pts)
        if n == 2 and segment['type'] == 'Straight':
            p0 = pts[0]
            p1 = pts[1]
            if not all_points:
                all_points.append(p0)
            else:
                # 연속된 점이 중복되지 않도록 처리
                if all_points[-1] != p0:
                    all_points.append(p0)
            all_points.append(p1)

        elif n == 4 and segment['type'] == 'Bezier':
            p0, c1, c2, p3 = pts
            curve_points = approximate_cubic_bezier(p0, c1, c2, p3, steps=10)
            if not all_points:
                all_points.extend(curve_points)
            else:
                if all_points[-1] == curve_points[0]:
                    all_points.extend(curve_points[1:])
                else:
                    all_points.extend(curve_points)
        else:
            raise ValueError("지원하지 않는 세그먼트 형식이거나 점 개수가 잘못되었습니다.")

    # 폐곡선 처리
    if all_points and (all_points[0] != all_points[-1]):
        all_points.append(all_points[0])
    return all_points

def compute_polygon_orientation(points):
    """
    points: (x, y) 좌표 리스트 (폐곡선), shoelace 공식으로 방향 판별
    """
    area = 0
    for i in range(len(points)-1):
        x0, y0 = points[i]
        x1, y1 = points[i+1]
        area += x0 * y1 - x1 * y0
    return 'ccw' if area > 0 else 'cw'

def extract_internal_offset_edges_by_segment(panel_edge_list, offset_dist):
    """
    panel_edge_list = [
      {"start_x": a, "start_y": b, "end_x": c, "end_y": d, "type": "Straight"},
      {"start_x": a, "start_y": b, "control1_x": e, "control1_y": f,
       "control2_x": g, "control2_y": h, "end_x": c, "end_y": d,
       "type": "Bezier Curve"},
      ...
    ]
    -> 내부 오프셋 된 "Straight" 세그먼트 리스트로 리턴 (예: [{"type":"Straight", "offset_points":[(x,y), (x',y')]}, ...])
    """

    # 1) panel_edge_list를 내부적으로 처리하기 쉽도록 기존 포맷으로 변환
    original_edges = []
    for seg in panel_edge_list:
        if seg["type"] == "Bezier Curve":
            # Bezier
            p0 = (seg["start_x"], seg["start_y"])
            c1 = (seg["control1_x"], seg["control1_y"])
            c2 = (seg["control2_x"], seg["control2_y"])
            p3 = (seg["end_x"], seg["end_y"])
            original_edges.append({
                'points': [p0, c1, c2, p3],
                'type': 'Bezier'
            })
        elif seg["type"] == "Straight":
            # Straight
            p0 = (seg["start_x"], seg["start_y"])
            p1 = (seg["end_x"], seg["end_y"])
            original_edges.append({
                'points': [p0, p1],
                'type': 'Straight'
            })
        else:
            raise ValueError("알 수 없는 segment type")

    # 2) 전체 외곽선 polyline 구하기
    outline_points = build_polyline_from_edges(original_edges)
    if len(outline_points) < 4:
        return []  # 무효한 폴리곤이라면 빈 리스트
    
    orientation = compute_polygon_orientation(outline_points)

    # 3) Shapely Polygon 생성
    polygon = Polygon(outline_points)
    if not polygon.is_valid:
        # self-intersection 등이 있다면 unary_union 등을 써서 정리
        polygon = polygon.buffer(0)

    if polygon.is_empty:
        return []

    # 4) 내부 오프셋 (negative buffer)
    #    offset_dist가 음수이든 양수이든, 내부 오프셋은 실제로 "음수 거리"로 buffer
    internal_polygon = polygon.buffer(-abs(offset_dist), resolution=16, join_style=1)
    # join_style=1 (round), =2(mitre), =3(bevel)
    # 상황에 맞게 옵션 변경 가능

    # 5) MultiPolygon 처리 (여러 조각으로 분할되었을 경우)
    if internal_polygon.is_empty:
        return []
    if internal_polygon.geom_type == 'MultiPolygon':
        # 가장 큰 면적만 선택
        internal_polygon = max(internal_polygon.geoms, key=lambda a: a.area)

    if internal_polygon.is_empty or internal_polygon.geom_type != 'Polygon':
        return []

    # 6) 내부 오프셋 결과 폴리곤의 exterior(바깥 경계) 좌표 추출
    offset_coords = list(internal_polygon.exterior.coords)
    # Shapely가 마지막 점 = 첫 점 형태로 닫혀 있음
    # -> 필요에 따라 마지막 점을 제거하거나 그대로 사용

    if len(offset_coords) < 2:
        return []

    # 7) 반환 형식으로 가공: "Straight" 세그먼트 모음
    #    => 예: [{"type":"Straight", "offset_points":[(x0,y0),(x1,y1)]},
    #            {"type":"Straight", "offset_points":[(x1,y1),(x2,y2)]}, ...]
    panel_offset_edges = []
    for i in range(len(offset_coords) - 1):
        pA = offset_coords[i]
        pB = offset_coords[i+1]
        panel_offset_edges.append({
            "type": "Straight",
            "offset_points": [pA, pB]
        })

    # 폴리곤이 닫혀 있으므로, 마지막 세그먼트(끝->처음) 연결을 원하는 경우 추가
    # 필요하다면 아래 주석 해제
    """
    pLast = offset_coords[-1]
    pFirst = offset_coords[0]
    panel_offset_edges.append({
        "type": "Straight",
        "offset_points": [pLast, pFirst]
    })
    """

    return panel_offset_edges

In [3]:
import os, sys

sys.path.append(os.path.dirname(os.getcwd()))

from env_constants import PYGARMENT_ROOT, DATASET_ROOT

In [8]:

GARMENT_ROOT_PATH = os.path.join(DATASET_ROOT, "GarmentCodeData_v2")
BODY_ROOT_PATH = os.path.join(DATASET_ROOT, "body_mesh")
MEAN_ALL_BODY_PATH = os.path.join(DATASET_ROOT, "neutral_body/mean_all.obj")

# random choice two number
BODY_TYPE = "default_body"
    
default_body_mesh = trimesh.load(MEAN_ALL_BODY_PATH)

garment_path_list = sorted(list(filter(
    os.path.isdir,
    glob(os.path.join(GARMENT_ROOT_PATH, "*", BODY_TYPE, "*"))
)))
len(garment_path_list)


132670

In [9]:
#len(garment_path_list[0])
garment_path = garment_path_list[0]
path_split= garment_path.split(os.sep)
garment_id= os.path.basename(garment_path)
print()


In [10]:
garment_path

'/media/hjp/05aba9a7-0e74-4e54-9bc9-5f11b9c4c757/GarmentCodeData/GarmentCodeData_v2/garments_5000_0/default_body/rand_00YONAPXZE'

In [8]:
# 임의로 5개만 테스트
generated = set()

#for IDX in tqdm(range(1)):
for IDX in tqdm(range(len(garment_path_list))):
    # random Fabric UUID 생성
    fabric_uuid = get_unique_random_string(generated)    
    random_hexa_code = generate_random_hex()
    garment_path = garment_path_list[IDX]
    path_split= garment_path.split(os.sep)
    garment_id= os.path.basename(garment_path)
    
    SPEC_FILE_PATH = os.path.join(garment_path,  f"{garment_id}_specification.json")
    pattern = pyg.pattern.wrappers.VisPattern(SPEC_FILE_PATH)
    
    drawn_pattern_list = list(map(
        lambda pannel_name : pattern._draw_a_panel(
            pannel_name, apply_transform=False, fill=True
        ),
        pattern.panel_order()
    ))
    
    panel_svg_path_dict = {
        panel_name : pattern._draw_a_panel(
            panel_name, apply_transform=False, fill=True
        )
        for panel_name in pattern.panel_order()
    }

    stitch_dict = {
        i : v for i, v in enumerate(pattern.pattern['stitches'])
    }

    # json의 rotation translation 추출을 위한 정보
    spec = load_json_file(SPEC_FILE_PATH)
    """
    # TODO: 2025.02.05 패널 중심점 대신 Mean Point 사용한 버전
    """
### 여기부터
    box_mesh = trimesh.load_mesh(
        os.path.join(garment_path, f"{garment_id}_boxmesh.ply"),
        process=False
    )

    idx_convert_map = np.array(v_id_map(box_mesh.vertices))


    # Read Vertex Label
    with open(os.path.join(garment_path, f"{garment_id}_sim_segmentation.txt"), "r") as f:
        segmentation = list(map(
            lambda x : x.strip(),
            f.readlines()
        ))
    stitch_vertex_mask_dict = {}
    for k in stitch_dict.keys():
        raw_mask = list(map(
            lambda x : True if f"stitch_{k}" in x.split(",") else False,
            segmentation
        ))
        base_mask = np.array(list(map(
            lambda idx : True if raw_mask[idx] else False,
            idx_convert_map
        )))
        stitch_vertex_mask_dict[k] = base_mask

    panel_vertex_mask_dict = {}
    for k in panel_svg_path_dict.keys():
        raw_mask = list(map(
            lambda x : True if x.startswith(k) else False,
            segmentation
        ))
        panel_vertex_mask_dict[k] = np.array(list(map(
            lambda idx : True if raw_mask[idx] else False,
            idx_convert_map
        )))

    panel_center_point_dict = {}
    for k in panel_vertex_mask_dict.keys():
        if len(panel_vertex_mask_dict[k]) > 0:  # panel_vertex_mask_dict[k]가 비어 있지 않은지 확인
            panel_center_point_dict[k] = np.mean(box_mesh.vertices[panel_vertex_mask_dict[k]], axis=0)
        else:
            panel_center_point_dict[k] = np.nan  # 빈 배열이면 NaN 또는 다른 적절한 값으로 설정
        
        
### 여기까지
    panel_fstart_fend_info = {}

    # 패널 리스트 (예: pattern.panel_order())
    panel_list = pattern.panel_order()

    # 각 패널에 대해 fstart, fend 계산
    for panel_name in panel_list:
        path = panel_svg_path_dict[panel_name][0]  # 패널에 해당하는 경로 정보 가져오기
        fstart_fend_results = calculate_fstart_fend(path) # TODO: 2024.01.19 14:33 panel_name 넣기
        panel_fstart_fend_info[panel_name] = fstart_fend_results  # 결과를 딕셔너리에 저장
        
    # transform data
    panel_datas = [process_path_data(drawn_pattern) for drawn_pattern in drawn_pattern_list]
    
    # for drawn_pattern in drawn_pattern_list:
    #     panel_datas.append(process_path_data(drawn_pattern))
    
    
    # panel_datas에 내가 리스트 하나 내부에 한 패널 의 엣지정보 담김.
    panel_names = list(pattern.panel_order())

    # Initialize the top-level JSON structure
    garment_data = {
        "FabricList": [],
        "GradingRuleTableList": [],
        "PatternList": [],
        "SymmetricDataList": [],
        "InstanceDataList": [],
        "SeamLinePairGroupList": [],
    }
    
    for panel_name, panel_data in zip(panel_names, panel_datas):
        # left sleeve_f or left sleeve_b 이면 돌리기
        if panel_name == "left_sleeve_f" or panel_name == "left_sleeve_b": #270도 돌려야할듯
            theta = np.pi * 3 / 2            
        elif panel_name == "right_sleeve_f" or panel_name == "right_sleeve_b": #270도 돌려야할듯
            theta = np.pi * 1 / 2
            # 패널 데이터 순회하며 좌표 회전 적용
        else:  
            theta = 0
            

        for idx, edge_data in enumerate(panel_data):
            if edge_data["type"] == "Arc":    
                data = ARC_TO_BEZIER(edge_data)
                panel_data[idx] = data.data_to_bezier_curve()
            else:
                pass
            
            # 필수 좌표 회전
            edge_data["start_x"], edge_data["start_y"] = rotate_point(edge_data["start_x"], edge_data["start_y"], theta)
            edge_data["end_x"], edge_data["end_y"] = rotate_point(edge_data["end_x"], edge_data["end_y"], theta)

            # 선택적 좌표 회전 (값이 존재하는 경우만)
            if "control1_x" in edge_data and "control1_y" in edge_data:
                edge_data["control1_x"], edge_data["control1_y"] = rotate_point(edge_data["control1_x"], edge_data["control1_y"], theta)
            
            if "control2_x" in edge_data and "control2_y" in edge_data:
                edge_data["control2_x"], edge_data["control2_y"] = rotate_point(edge_data["control2_x"], edge_data["control2_y"], theta)
        
        # 한패널에 대해서 데이터 변환 완료.
        panel_data = process_negative_columns(panel_name, panel_data)
                                
        panel_batch_point = matching_point(panel_center_point_dict[panel_name]) # batch point 이름 받기
        
        panel_data_json = {
            "Name": f"{garment_id}_{panel_name}", # garment_id 추가함.
            "fGrainlineAngle": 0.0,
            "ID": f"{garment_id}_{panel_name}", # garment_id 추가함.
            "strSuperImposeSide": "None",
            "CurrentFabricUUID": fabric_uuid,
            "IsClosed": False,
            "InternalLineList": [],
            "ButtonHeadList": [],
            "ButtonHoleList": [],
            "AnnotationList": [],
            "NotchList": [],
            "IsHalfSymmetric": False,
            "ShapeInfo": {
                "IsSlashed": False,
                "LineList": [],
                "HalfSymmetryPointIDMap": {},
                "HalfSymmetryLineIDMap": {}
            },
            "ArrangementPointDataMap": {
                    "PointName": panel_batch_point,
                    "fOffSetX": 0.0,
                    "fOffSetY": 0.05,
                    "fAngle": 180.0
            }
        }
        # InstanceDataList에 json에 추가
        instance_data = {
            "OriginPatternID": panel_name,  # 패널 이름을 OriginPatternID로 사용
            "InstancePatternIDArray": []   # 빈 리스트로 추가
        }
        
        garment_data["InstanceDataList"].append(instance_data)
        # 패널별 좌표 ID 초기화
        coordinate_id_map = {}
        point_counter = 0  # 패널 내 좌표 카운터
############
        # Generate LineList and PointList for the panel
        for edge_idx, segment in enumerate(panel_data):
            line_id = f"{panel_name}_edge_{edge_idx}"
            point_list = [] 
            
            if segment["type"] == "Bezier Curve":
                row_type = ["Straight", "Bezier Curve", "Bezier Curve", "Straight"]
                col_x = ["start_x", "control1_x", "control2_x", "end_x"]
                col_y = ["start_y", "control1_y", "control2_y", "end_y"]
                idx = 0
                
                for num_x, num_y in zip(col_x, col_y):
                    x, y = segment[num_x], segment[num_y]
                    key = (x, y)
                    if key not in coordinate_id_map:
                        coordinate_id_map[key] = f"{panel_name}_point_{point_counter}"
                        point_counter += 1
                    point_id = coordinate_id_map[key]
                
                    point_list.append({
                        "ID": point_id, 
                        "PointType": row_type[idx],
                        "Position": {"x": x, "y": y},
                        "GradingRuleID": -1
                    })
                    idx +=1
                    
            elif segment["type"] == "Straight":
                col_x = ["start_x", "end_x"]
                col_y = ["start_y", "end_y"]
                for num_x, num_y in zip(col_x, col_y):  # Skip 'type' and go by pairs
                    x, y = segment[num_x], segment[num_y]
                    key = (x, y)
                    if key not in coordinate_id_map:
                        coordinate_id_map[key] = f"{panel_name}_point_{point_counter}"
                        point_counter += 1
                    point_id = coordinate_id_map[key]
                    
                    point_list.append({
                        "ID": point_id, 
                        "PointType": "Straight",
                        "Position": {"x": x, "y": y},
                        "GradingRuleID": -1
                    })
                     
            # Add Line to LineList
            line = {"ID": line_id, "PointList": point_list}
            panel_data_json["ShapeInfo"]["LineList"].append(line)
            
        garment_data["PatternList"].append(panel_data_json)
        ######################################################### 좀 확인.
        offset_edges_by_panel = extract_internal_offset_edges_by_segment(panel_data, offset_dist= -2.0)
        internal_line_list = []
        internal_coordinate_id_map = {}  # 내부 포인트 좌표와 ID를 매핑하기 위한 딕셔너리
        internal_point_counter = 0
        internal_line_counter = 0
        #print(offset_edges_by_panel)
        line_list = [] #246에 있던걸 여기로 옮겨봄.
        for panel_idx, edge_panel in enumerate(offset_edges_by_panel):
            #line_list = [] 
            # 각 panel의 각 엣지(라인)에 대해 처리
            pt_list = []
            pts = edge_panel['offset_points']
            if not pts:
                continue
            if edge_panel['type'] == 'Straight':
                # Straight의 경우 모든 점의 PointType은 "Straight"
                for p in pts:
                    coord = (p[0], p[1])
                    if coord not in internal_coordinate_id_map:
                        internal_coordinate_id_map[coord] = f"{panel_name}_internal_point_{internal_point_counter}"
                        internal_point_counter += 1
                    point_id = internal_coordinate_id_map[coord]
                    point_dict = {
                        "ID": point_id,
                        "PointType": "Straight",
                        "Position": {"x": p[0], "y": p[1]},
                        "GradingRuleID": -1
                    }
                    pt_list.append(point_dict)
            elif edge_panel['type'] == 'Bezier':
                # Bezier의 경우, 첫점과 마지막 점은 "Straight", 중간은 "Polyline"
                for i, p in enumerate(pts):
                    point_type = "Straight" if (i == 0 or i == (len(pts) - 1)) else "Polyline"
                    coord = (p[0], p[1])
                    if coord not in internal_coordinate_id_map:
                        internal_coordinate_id_map[coord] = f"{panel_name}_internal_point_{internal_point_counter}"
                        internal_point_counter += 1
                    point_id = internal_coordinate_id_map[coord]
                    point_dict = {
                        "ID": point_id,
                        "PointType": point_type,
                        "Position": {"x": p[0], "y": p[1]},
                        "GradingRuleID": -1
                    }
                    pt_list.append(point_dict)
            else:
                continue

            line_dict = {
                "ID": f"{panel_name}_internal_line_{internal_line_counter}",
                "PointList": pt_list
            }
            line_list.append(line_dict)
            internal_line_counter += 1

            # 예제에서는 첫번째 패널은 닫힌 모양(closed), 두번째는 열린 모양(open)으로 가정
        panel_dict = {
            "LineList": line_list,
            "ID": f"{panel_name}_internal_line",
            "ShapeType": 1, #if panel_idx == 0 else 2,
            "IsClosed": True, #if panel_idx == 0 else False,
            "FoldData": {
                "iAngle": 90,
                "iStrength": 5,
                "bRenderFolded": False
            }
        }
        
        internal_line_list.append(panel_dict)

        panel_data_json["InternalLineList"] = internal_line_list

        
    garment_data["SeamLinePairGroupList"] = []
    fabric_group = {
        "FabricName" : "Default Fabric",
        "FabricType" : "None",
        "FabricContent" : "None",
        "strBaseColorHexCode" : random_hexa_code,
        "FabricUUID" : fabric_uuid
        }
    # SeamLinePairGroupList 생성 루프
    for idx, value in enumerate(stitch_dict.values()):
        # "Name" 생성
        group_name = f"{garment_id}_SeamLineGroup_{idx}"

        # value에서 첫 번째와 두 번째 ShapeID 정보 추출
        
        first_info = value[0]
        second_info = value[1]
            
        # 정보 저장
        first_panel = first_info["panel"] # 패널 정보
        first_edge = first_info["edge"]
        first_shape_id = f"{garment_id}_{first_panel}"
        
        second_panel = second_info["panel"]
        second_edge = second_info["edge"]
        second_shape_id = f"{garment_id}_{second_panel}" #TODO: 이부분 제외 '_edge_{second_edge}"'
        
        # 첫 번째 fStart와 fEnd 추출
        first_fstart_fend = next(
            (item for item in panel_fstart_fend_info[first_panel] if item["edge_id"] == first_edge),
            None
        )
        # 두 번째 fStart와 fEnd 추출
        second_fstart_fend = next(
            (item for item in panel_fstart_fend_info[second_panel] if item["edge_id"] == second_edge),
            None
        )
        
        # first, second panel 정보 넣음
        if first_fstart_fend is not None:
            first_fstart_fend["panel"] = first_panel
        first_point = first_fstart_fend["edge"]
        
        if second_fstart_fend is not None:
            second_fstart_fend["panel"] = second_panel
        second_point = second_fstart_fend["edge"]
        
        first_direction = True
        second_direction = False
        
        # direction 결정 뒤에 정해 놓는 fstart, fend
        if first_direction == False:
            first_fstart, first_fend = first_fstart_fend["fend"], first_fstart_fend["fstart"]
        else:
            first_fstart, first_fend = first_fstart_fend["fstart"], first_fstart_fend["fend"]        
            
        if second_direction == False:
            second_fstart, second_fend = second_fstart_fend["fend"], second_fstart_fend["fstart"]
        else:
            second_fstart, second_fend = second_fstart_fend["fstart"], second_fstart_fend["fend"]
        

        # JSON 구조 생성
        seam_line_pair_group = {
            "Name": group_name,
            "bIsTurned": False,
            "PairList": [
                {
                    "First": {
                        "ShapeID": first_shape_id,
                        "LengthParam": {
                            "fStart": first_fstart,
                            "fEnd": first_fend
                        },
                        "Direction": first_direction
                    },
                    "Second": {
                        "ShapeID": second_shape_id,
                        "LengthParam": {
                            "fStart": second_fstart,
                            "fEnd": second_fend
                        },
                        "Direction": second_direction 
                    }
                }
            ],
            "FoldData": {
                "iAngle": 270,
                "iStrength": 5
            }
        }

        # SeamLinePairGroupList에 추가
        garment_data["SeamLinePairGroupList"].append(seam_line_pair_group)

    garment_data["FabricList"].append(fabric_group)    

    panel_index_mapping = {name: idx for idx, name in enumerate(panel_names)}
    
    stitch_info = convert_stitch_dict(stitch_dict, panel_index_mapping)
    
    panel_dict = {"panel_names" : panel_names}
    DIR = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/ADD_INTERNAL_LINE_DATA"
    DIR_NAME = f"{path_split[-3]}_{garment_id}"
    JSON_PATH = os.path.join(DIR, DIR_NAME)
    os.makedirs(os.path.join(DIR, DIR_NAME), exist_ok=True)
    # CLO JSON DATA FILE
    json_output_file = f"{JSON_PATH}/Clo_Garment_{path_split[-3]}_{garment_id}.json"
    # STITCH DATA FILE
    stitch_output_file = f"{JSON_PATH}/Clo_Stitch_{path_split[-3]}_{garment_id}.json"
    panel_name_file = f"{JSON_PATH}/Clo_Panel_Name_{path_split[-3]}_{garment_id}.json"

    save_json(garment_data, json_output_file)
    save_json(stitch_info, stitch_output_file)
    save_json(panel_dict, panel_name_file)
    

"""
# 여기서 부터 0218 Ver
        # Generate LineList and PointList for the panel
        offset_edges_by_panel = extract_internal_offset_edges_by_segment(panel_data, offset_dist=-2.0)
        internal_line_list = []
        internal_point_counter = 0
        internal_line_counter = 0
        for panel_idx, panel in enumerate(offset_edges_by_panel):
            line_list = []
            # 각 panel의 각 엣지(라인)에 대해 처리
            for edge in panel:
                pt_list = []
                pts = edge['offset_points']
                if not pts:
                    continue
                if edge['type'] == 'Straight':
                    # Straight의 경우 두 점 모두 PointType은 "Straight"
                    for p in pts:
                        point_dict = {
                            "ID": f"{panel_name}_internal_point_{internal_point_counter}",
                            "PointType": "Straight",
                            "Position": {"x": p[0], "y": p[1]},
                            "GradingRuleID": -1
                        }
                        pt_list.append(point_dict)
                        internal_point_counter += 1                        
                elif edge['type'] == 'Bezier':
                    # Bezier의 경우, 첫점과 마지막 점은 "Straight", 중간은 "Polyline"
                    for i, p in enumerate(pts):
                        if i == 0 or i == (len(pts) - 1):
                            point_type = "Straight"
                        else:
                            point_type = "Polyline"
                        point_dict = {
                            "ID": f"{panel_name}_internal_point_{internal_point_counter}",
                            "PointType": point_type,
                            "Position": {"x": p[0], "y": p[1]},
                            "GradingRuleID": -1
                        }
                        pt_list.append(point_dict)
                        internal_point_counter += 1
                else:
                    continue

                line_dict = {
                    "ID": f"{panel_name}_internal_line_{internal_line_counter}",
                    "PointList": pt_list
                }
                line_list.append(line_dict)
                internal_line_counter += 1
            
            # 예제에서는 첫번째 패널은 닫힌 모양(closed)으로, 두번째는 열린 모양(open)으로 가정
            panel_dict = {
                "LineList": line_list,
                "ID": f"{panel_name}_internal_line_{internal_line_counter}",
                "ShapeType": 1 if panel_idx == 0 else 2,
                "IsClosed": True if panel_idx == 0 else False,
                "FoldData": {
                    "iAngle": 180,
                    "iStrength": 5,
                    "bRenderFolded": False
                } if panel_idx == 0 else None
            }
            # FoldData가 None이면 key 삭제
            if panel_dict["FoldData"] is None:
                panel_dict.pop("FoldData")
            internal_line_list.append(panel_dict)            
"""       

  0%|          | 0/132670 [00:00<?, ?it/s]

  0%|          | 12/132670 [00:05<17:32:19,  2.10it/s]


AttributeError: 'str' object has no attribute 'area'

In [54]:
for idx ,a in enumerate(ccc):
    print(idx, a)

0 {'type': 'Straight', 'offset_points': [(152.76047258650246, 2.0000000000000093), (152.76047258650246, 150.6834480704392)]}
1 {'type': 'Bezier', 'offset_points': [(152.76047258650246, 150.6834480704392), (131.59681008834204, 149.94766268068156), (111.86451363858212, 144.76886104257792), (94.9210461500442, 137.03404737522783), (80.10784421180735, 127.29767152016503), (66.79052616552418, 116.10854351168467), (54.351196636190835, 104.02767457547395), (42.19158367022578, 91.64997423715832), (42.15493994734612, 91.61362368953796), (29.68790729102361, 79.56150789299878), (29.576142023891464, 79.46129766695084), (16.195433057021496, 68.34092347732316), (15.999401862275628, 68.19721211160986), (1.9999999999999964, 59.1551215587902)]}
2 {'type': 'Straight', 'offset_points': [(1.9999999999999964, 59.1551215587902), (2.0, 2.0)]}
3 {'type': 'Straight', 'offset_points': [(2.0, 2.0), (152.76047258650246, 2.0000000000000093)]}
4 [{'type': 'Straight', 'offset_points': [(152.76047258650246, 2.00000000

In [393]:
theta

0

In [317]:
for panel_name, panel_data in zip(panel_pattern_list, panel_datas):
    print(panel_name, panel_data[0]["type"])

right_sleeve_b Straight
sl_right_cuff_b Straight
right_sleeve_f Straight
sl_right_cuff_f Straight
right_btorso Bezier Curve
right_ftorso Straight
right_hood Bezier Curve
skirt_back Bezier Curve
skirt_front Straight
wb_back Straight
wb_front Straight
left_hood Bezier Curve
left_btorso Straight
left_ftorso Straight
left_sleeve_b Straight
sl_left_cuff_b Straight
left_sleeve_f Straight
sl_left_cuff_f Straight


In [236]:
coordinate_id_map

{(np.float64(0.0), np.float64(0.0)): 'left_ftorso_point_0',
 (np.float64(72.02177611667605), np.float64(0.0)): 'left_ftorso_point_1',
 (np.float64(81.89112611667606),
  np.float64(108.44089484057153)): 'left_ftorso_point_2',
 (np.float64(91.76047611667609), np.float64(0.0)): 'left_ftorso_point_3',
 (np.float64(211.83020000000002), np.float64(0.0)): 'left_ftorso_point_4',
 (np.float64(221.69955000000004),
  np.float64(120.98786645750994)): 'left_ftorso_point_5',
 (np.float64(110.52948438217182),
  np.float64(136.5811164575099)): 'left_ftorso_point_6',
 (np.float64(221.69955000000004),
  np.float64(152.1743664575099)): 'left_ftorso_point_7',
 (np.float64(207.91925458975794),
  np.float64(191.92864457542413)): 'left_ftorso_point_8',
 (np.float64(145.0830432187303),
  np.float64(191.92864457542413)): 'left_ftorso_point_9',
 (np.float64(145.0830432187303),
  np.float64(348.68122457542415)): 'left_ftorso_point_10',
 (np.float64(91.9598), np.float64(369.7975811246332)): 'left_ftorso_point_11'

In [117]:
#sum(calculate_edge_length(edge_info) for edge_info in path)
# for edge in path:
#     print(edge)

for edge_info in path:
    print(calculate_edge_length(edge_info))
    
#path[3].length()

Line(start=0j, end=(18.117238950508963+0j))
18.117238950508963
CubicBezier(start=(18.117238950508963+0j), control1=(18.008224307512876+9.975699377309166j), control2=(10.480849152257248+11.974606507809122j), end=(7.091744534163308+18.207114421941466j))
21.915324602613726
Line(start=(7.091744534163308+18.207114421941466j), end=18.207114421941466j)
7.091744534163308
Line(start=18.207114421941466j, end=0j)
18.207114421941466


In [115]:
edge_info.__class__.__name__


'Line'

In [174]:
panel_data_list[0]

{'type': 'Straight',
 'start_x': np.float64(154.76047258650246),
 'start_y': np.float64(9.476345869379597e-15),
 'end_x': np.float64(154.76047258650246),
 'end_y': np.float64(153.9965310793262)}

In [175]:
panel_data_list[0].keys()

dict_keys(['type', 'start_x', 'start_y', 'end_x', 'end_y'])

In [57]:
json_file_path = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/garment_without_arc_list.json"

def load_json_file(file_path):
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

json_data = load_json_file(json_file_path)
len(json_data)

30427

In [70]:
# len(os.listdir("./NO_ARC_CLO_JSON"))
os.listdir("./NO_ARC_CLO_JSON")[0]

'Clo_Garment_garments_5000_13_rand_ZP4QMHSC6J.json'

In [8]:
# JSON 파일 읽고 가지고 테스트
PATH_TST = "./NO_ARC_CLO_JSON/garments_5000_0_rand_0ADI18HWRF/Clo_Garment_garments_5000_0_rand_0ADI18HWRF.json"

with open(PATH_TST, "r") as f:
    data = json.load(f)

data

In [8]:
GARMENT_PATH_1 = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/uni_test_test/garments_5000_0_rand_0ADI18HWRF/Clo_Garment_garments_5000_0_rand_0ADI18HWRF.json"
GARMENT_PATH_2 = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/uni_test_test/garments_5000_0_rand_0BVS2KDJ6H/Clo_Garment_garments_5000_0_rand_0BVS2KDJ6H.json"

uni_data = unify_clo_json_data(GARMENT_PATH_1, GARMENT_PATH_2)

In [9]:
STITCH_PATH_1 = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/uni_test_test/garments_5000_0_rand_0ADI18HWRF/Clo_Stitch_garments_5000_0_rand_0ADI18HWRF.json"
STITCH_PATH_2 = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/uni_test_test/garments_5000_0_rand_0BVS2KDJ6H/Clo_Stitch_garments_5000_0_rand_0BVS2KDJ6H.json"


stitch_data, garment_list = stitch_data_unify(STITCH_PATH_1, STITCH_PATH_2)



In [11]:
SAVE_PATH = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/uni_test_test"
with open(f"{SAVE_PATH}/Clo_Garment_Unify.json", "w") as file:
    json.dump(uni_data, file, indent=4)
with open(f"{SAVE_PATH}/Clo_Stitch_Unify.json", "w") as file:
    json.dump(stitch_data, file, indent=4)
with open(f"{SAVE_PATH}/Clo_garment_id.json", "w") as file:
    json.dump(garment_list, file, indent=4)

In [ ]:
def stitch_data_unify(Clo_stitch_path, Clo_stitch_path_to_add):
    with open(Clo_stitch_path, "r") as f:
        stitch_data_1 = json.load(f)
    #stitch_data_1
    with open(Clo_stitch_path_to_add, "r") as f:
        stitch_data_2 = json.load(f)
    
    max_panel_1 = max(d["panel"] for d in stitch_data_1)
    offset = max_panel_1 + 1
    for item in stitch_data_2:
        item["panel"] += offset
    return stitch_data_1.extend(stitch_data_2)

In [82]:
stitch_data_unify()

In [ ]:
import svgpathtools
from svgpathtools import Path, Line
from matplotlib.colors import CenteredNorm
import time
def is_point_inside_path(path: Path, point: complex) -> bool:
    """
    Determine if a point is inside the path using ray casting.
    """
    ray_end = point + 1000 + 0j  # Cast ray in positive x direction
    intersections = 0
    for segment in path:
        ray = Line(point, ray_end)
        crossings = segment.intersect(ray)
        # crossings returns list of (t1, t2) tuples
        # t1 is parameter for first curve, t2 for second curve
        # We only care about t1 values between 0 and 1
        intersections += sum(1 for t1, t2 in crossings if 0 <= t1 <= 1)
    return intersections % 2 == 1
def compute_signed_distance_grid(
    path: svgpathtools.Path,
    n_samples: int = 1000,
    grid_size: int = 200,
    SDF_DOMAIN_SIZE: float = 2,
    ZOOM_OUT_FACTOR: float = 1.2,
    K_NEIGHBORS: int = 5
) -> torch.Tensor:
    """
    Compute signed distance field for an SVG path.
    args :
        path : svgpathtools.Path
            unnormalized closed path
        n_samples : int, number of samples along the path
        img_size : int, size of the grid
    """
    edge_lengths = [segment.length() for segment in path]
    total_length = sum(edge_lengths)
    # Distribute points proportionally to edge lengths
    points_per_edge = [
        max(int(n_samples * length / total_length), 10)  # Ensure minimum 10 points per edge
        for length in edge_lengths
    ]
    edge_points = []
    for edge_idx, segment in enumerate(path):
        t_vals = torch.linspace(0, 1, points_per_edge[edge_idx])
        edge_samples = torch.tensor([
            [segment.point(t.item()).real, segment.point(t.item()).imag]
            for t in t_vals
        ])
        edge_points.append(edge_samples)
    # Combine all points and create edge index mapping
    boundary_points = torch.cat(edge_points, dim=0)
    edge_indices = torch.cat([
        torch.full((points_per_edge[i],), i, dtype=torch.int64)
        for i in range(len(path))
    ])
    # get bounding box of the path
    xmin, xmax, ymin, ymax = path.bbox()
    x_center = (xmin + xmax) / 2
    y_center = (ymin + ymax) / 2
    x_scale = xmax - xmin
    y_scale = ymax - ymin
    scale_factor = max(x_scale, y_scale) * ZOOM_OUT_FACTOR / SDF_DOMAIN_SIZE
    boundary_points_normalized = (
        boundary_points - torch.tensor([x_center, y_center])
    ) / scale_factor
    # get grid of evaluation points
    coords = torch.stack(torch.meshgrid(
        torch.linspace(-1, 1, grid_size),
        torch.linspace(-1, 1, grid_size)
    ), dim=-1).reshape(-1, 2)[:, [1, 0]].float()
    # Compute unsigned distances
    coords_expanded = coords.unsqueeze(1)
    points_expanded = boundary_points_normalized.unsqueeze(0)
    distances = torch.norm(coords_expanded - points_expanded, dim=2)
    unsigned_distances, min_indices = torch.min(distances, dim=1)
    k_distances, k_indices = torch.topk(
        distances, k=K_NEIGHBORS, dim=1, largest=False
    )
    k_edge_indices = edge_indices[k_indices]
    weights = 1.0 / (k_distances + 1e-6)
    n_edges = len(path)
    one_hot = torch.zeros(
        k_edge_indices.shape[0], K_NEIGHBORS, n_edges
    )
    one_hot.scatter_(2, k_edge_indices.unsqueeze(-1), 1)
    # Apply weights to votes
    weighted_votes = one_hot * weights.unsqueeze(-1)
    # Sum votes for each edge
    edge_votes = weighted_votes.sum(dim=1)
    # Get edge with maximum votes
    closest_edge_indices = edge_votes.argmax(dim=1)
    # closest_edge_indices = edge_indices[min_indices]
    # For sign computation, transform coords back to original space
    signs = torch.tensor([
        -1 if is_point_inside_path(
            path, complex(x.item(), y.item())
        ) else 1
        for x, y in coords * scale_factor + torch.tensor([x_center, y_center])
    ])
    signed_distances = unsigned_distances * signs
    return (
        signed_distances.reshape(grid_size, grid_size),
        closest_edge_indices.reshape(grid_size, grid_size),
        scale_factor
    )
IMG_SIZE = 256
N_SAMPLES = 4000
PATH_IDX = 2
panel_name = pattern.panel_order()[PATH_IDX]
panel_svg_path = panel_svg_path_dict[panel_name][0]
fig, ax = plt.subplots()
ax.set_title(f"{panel_name}")
plot_panel_info(
    ax, panel_name,
    panel_svg_path_dict,
    stitch_dict,
    N_SAMPLES=1000
)
plt.show()
pprint(panel_svg_path)
sd_grid, edge_indices_grid, scale_factor = compute_signed_distance_grid(
    panel_svg_path,
    n_samples=N_SAMPLES,
    grid_size=IMG_SIZE
)
print("SCALE FACTOR : ", scale_factor)
THRESHOLD = 0.005
grid_np = sd_grid.cpu().numpy()
boundary_mask = np.logical_and(grid_np < THRESHOLD, grid_np > -THRESHOLD)
boundary_y, boundary_x = np.where(boundary_mask)
plt.imshow(edge_indices_grid.cpu().numpy(), cmap='Set3')
plt.colorbar(label='Edge Index')
plt.scatter(boundary_x, boundary_y, c='r', s=0.01)
plt.axis('equal')
plt.show()
plt.imshow(grid_np, cmap='RdBu', norm=CenteredNorm())
plt.colorbar(label='Signed Distance')
plt.axis('equal')
# plt.scatter(boundary_x, boundary_y, c='r', s=0.01)
plt.show()

In [71]:
PATH = "/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/Unify_Clo_json_data"

path123 = os.path.join(PATH, "case_2")
print(path123)

/media/hjp/db6095ca-a560-4c3a-90ad-b667ec189671/REFERENCES/3D_VTO/GarmentCode/GarmentCode/ANALYSIS_josef/Unify_Clo_json_data/case_2


In [ ]:
# ?? ??? ?? ??

# ??? ??? ?? ??? ???? ? ??.
import os
import json

import import_api
import utility_api
import fabric_api
import pattern_api
import export_api
import ApiTypes
import time
import os
import glob
import random

# definition
def open_json(path):
    with open(path[0], "r") as f:
        data = json.load(f)    
    return data
    
def garment_info_json(path):
    with open(path, "r", encoding="utf-8") as file:
        info = json.load(file)
    return info

def seam_taping(stitch_info):
    for value in stitch_info:
        panel_num = value['panel']
        edge_num = value['edge']
        pattern_api.SetPatternPieceSeamtaping(panel_num, edge_num, True)
        pattern_api.SetPatternPieceSametapingWidth(panel_num, edge_num , 2)

def random_fabric_path(fabric_path):
    num1, num2 = random.sample(range(0, len(os.listdir(fabric_path)) - 1), 2)
    fabric_paths = [os.path.join(fabric_path, path) for path in os.listdir(fabric_path)]
    return fabric_paths[num1], fabric_paths[num2]

def adding_outfit_fabric(end_num_index):
    for num in range(0, end_num_index, 1):
        fabric_api.AssignFabricToPattern( 2, num, 2)
        pattern_api.SetArrangementShapeStyle(num, "Flat")


def adding_top_bottom_fabric(end_num_index, end_num_index_2):
    for num in range(0, end_num_index, 1):
        fabric_api.AssignFabricToPattern( 2, num, 2)
        pattern_api.SetArrangementShapeStyle(num, "Flat")
#        fabric_api.AssignFabricToPattern( 1, num, 1)
    for second_num in range(end_num_index, end_num_index + end_num_index_2, 1):
        fabric_api.AssignFabricToPattern( 3, second_num, 3)#2)
        pattern_api.SetArrangementShapeStyle(num, "Flat")
        
ieo_option = ApiTypes.ImportExportOption()
ieo_option.bExportGarment = True
ieo_option.bExportAvatar = False
ieo_option.bSingleObject = True # single or Multiple
#ieo_option.weldType = WELD_TYPE
ieo_option.bThin = False
ieo_option.bSaveInZip = False
ieo_option.bMetaData = True
# ??? load

TYPE = ["outfit", "top_bottom"]

avtfile_path = r"D:\VTO2025\DATASETs\Ours1\CLO_JSON_DATAs\Mia_avt_revise.avt" # ddung miya 로 변경
custom_point_path = r"D:/VTO2025/DATASETs/Ours1/CLO_JSON_DATAs/View_Point"
#pose_path = "D:/VTO2025/DATASETs/Ours1/CLO_JSON_DATAs/Pose" #  1 pose만 적용 한 채로 
pose_path = "./dir.pose"
DATA_DIR = r"D:\VTO2025\DATASETs\Ours1\CLO_JSON_DATAs" # 여기에 submit용 폴더 하나 만들어야함.
FABRIC_PATH = r"D:\VTO2025\DATASETs\Ours1\CLO_JSON_DATAs\Fabric_CLO"
Garment_info_path = r"D:\VTO2025\DATASETs\Ours1\CLO_JSON_DATAs\info"
standard_pose_path = "/Users/seph/Downloads/Front_View.zcmr"

pass_garment_list = ["garments_5000_333"] # debug

for type in TYPE:
    DATA_TYPE_DIR = os.path.join(DATA_DIR, f"{type}_CLO_DATASET") # outfit?? top_bottom?? ??
    for path in os.listdir(DATA_TYPE_DIR): 
        # ?? texture root ???? 
        GARMENT_NUM_DIR = os.path.join(DATA_TYPE_DIR, path) # train test valid
        
        for garment_num in os.listdir(GARMENT_NUM_DIR):# garment_5000_?
            if garment_num.startswith(pass_garment_list):
                # didn't need to do anything
                continue
            else:
                GARMENT_RAND_DIR = os.path.join(GARMENT_NUM_DIR, garment_num)
                for garment_dir in os.listdir(GARMENT_RAND_DIR): # rand_????_rand_????
                    
                    GARMENT_DIR = os.path.join(GARMENT_RAND_DIR, garment_dir) 
                    garment_dir_id = os.path.basename(GARMENT_DIR)
                    
                    garment_ids = garment_dir_id.split("_") # extract top_id, bottom_id
                    
                    top_id, bottom_id = (garment_ids[0]+"_"+garment_ids[1]), (garment_ids[2]+"_"+garment_ids[3])

                    target_top_base_id = top_id
                    target_bottom_base_id = bottom_id
                    info = garment_info_json(os.path.join(Garment_info_path, f"{type}_{path}_transformed_data.json"))

                    combined_list = info[garment_num]
                    
                    if not target_top_base_id == target_bottom_base_id:
                        # {aa}_transformed_data.json #train test valid
                        matching_item = next(
                            (item for item in combined_list 
                            if item.get("top_base_id") == target_top_base_id and item.get("bottom_base_id") == target_bottom_base_id),
                            None
                        )
                        top_panel = matching_item["top_panel_count"]
                        bottom_panel = matching_item["bottom_panel_count"]
                        combi_num = 2

                    else:
                        #outfit_{aa}_transformed_data.json #train test valid
                        matching_item = next(
                            (item for item in combined_list 
                            if item.get("outfit_base_id") == target_top_base_id),
                            None
                        )
                        outfit_panel = matching_item["outfit_panel_count"]
                        combi_num = 1


                    fabric_path_1, fabric_path_2 = random_fabric_path(FABRIC_PATH)
                    
                    if path.startswith(".DS_Store"):
                        continue
                        

                    export_zpac = os.path.join(GARMENT_DIR, f"{garment_ids}.zpac")
                    JSON_PATH = glob.glob(os.path.join(GARMENT_DIR, "Clo_Garment_*.json"))
                    STITCH_PATH = glob.glob(os.path.join(GARMENT_DIR, "Clo_Stitch_*.json"))
                        
                    #print(STITCH_PATH)
                    stitch_json = open_json(STITCH_PATH)
                    import_api.ImportFile(avtfile_path)
                    pattern_api.ImportPatternJSON(JSON_PATH[0]) # garment import
                    export_api.ExportZPac(export_zpac)
                        
                    import_api.ImportFile(avtfile_path)
                    import_api.ImportFile(export_zpac)
                #        import_api.ImportFile(export_zpac, add_option)
                    fabric_api.AddFabric(fabric_path_1)
                    fabric_api.AddFabric(fabric_path_2)
                    # ???? fabirc? pattern? ?????
                    if combi_num == 2 :
                        adding_top_bottom_fabric(top_panel, bottom_panel)
                    if combi_num == 1 :
                        adding_outfit_fabric(outfit_panel)
                    else :
                        print("garment_id_list is not 1 or 2 ????.")
                        # ????
                        
                    seam_taping(stitch_json)
                        
                    export_api.ExportSnapshot3D(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_before_drape.png"))
                    export_api.ExportOBJ(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_before_drape.obj"),ieo_option)

                    utility_api.Simulate(200)
                        # ???? ? obj ??
                    export_api.ExportSnapshot3D(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_after_drape.png"))
                    export_api.ExportOBJ(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_after_drape.obj"), ieo_option)

                    utility_api.NewProject()
                    
                    import_api.ImportFile(avtfile_path)
                    import_api.ImportFile(export_zpac)
                                                    
                #        import_api.ImportFile(avtfile_path) # avatar import
                #        import_api.ImportFile(export_zpac)
                        
                    fabric_api.AddFabric(fabric_path_1)
                    fabric_api.AddFabric(fabric_path_2)
                            
                    # ???? fabirc? pattern? ?????
                    if combi_num == 2 :
                        adding_top_bottom_fabric(top_panel, bottom_panel)
                    if combi_num == 1 :
                        adding_outfit_fabric(outfit_panel)
                    else :
                        print("garment_id_list is not 1 or 2 ????.")
                    # ????                    
                        
                    seam_taping(stitch_json)             
                    pose_file = pose_path
                    pose_name = os.path.basename(pose_file)
                    if pose_path.startswith(".DS_Store"):
                        continue
                    import_api.ImportPose(pose_file)
                        
                    for custom_point in os.listdir(custom_point_path):
                        custom_point_file = os.path.join(custom_point_path, custom_point)
                        import_api.ImportFile(custom_point_file) # view import
                        utility_api.Simulate(200)
                        #export_api.ExportSnapshot3D(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_{pose_name}_{custom_point}_pose.png"))
                        export_api.ExportRenderingImage(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_{pose_name}_{custom_point}_rendering.png"))
                    import_api.ImportFile(standard_pose_path)
                    export_api.ExportOBJ(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_{pose_name}_after_pose.obj"),ieo_option)                            
                    # for pose in os.listdir(pose_path):
                    #     if pose.startswith(".DS_Store"):
                    #         continue
                    #     pose_file = os.path.join(pose_path, pose)
                    #     if pose.startswith(".DS_Store"):
                    #         continue
                    #     import_api.ImportPose(pose_file)
                            
                    #     for custom_point in os.listdir(custom_point_path):
                    #         custom_point_file = os.path.join(custom_point_path, custom_point)
                    #         import_api.ImportFile(custom_point_file) # view import
                    #         utility_api.Simulate(200)
                    #         export_api.ExportSnapshot3D(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_{pose}_{custom_point}_pose.png"))
                    #     export_api.ExportOBJ(os.path.join(GARMENT_DIR,f"{top_id}_{bottom_id}_{pose}_after_pose.obj"),ieo_option)

                utility_api.NewProject()















